# MBAI 448 | Week 7 Walkthrough: Aligned AI - Retrieval Augmented Generation (RAG)

## Business Goal / Case Statement
TBD

## Assignment Context

**Relevant Industry and/or Business Function:** TBD

**Description:**

TBD

## Assignment Tools

**Platform:** [Google Colab](https://colab.research.google.com/)<br />
**IDE:** [Jupyter Notebooks](https://jupyter.org/)<br />
**Language:** [Python](https://www.python.org/)<br />
**Packages/Libraries:** [HuggingFace](https://huggingface.co/docs)<br />

## AI/ML Task(s)

Representation<br />

## Algorithmic Technique(s)

[RAG](https://huggingface.co/docs/transformers/en/model_doc/rag)

## The Data

**Dataset Name:** <code>[llmware/rag_instruct_test_dataset2_financial_0.1](https://huggingface.co/datasets/llmware/rag_instruct_test_dataset2_financial_0.1)</code><br>
**Data Location:** <code>https://huggingface.co/datasets/llmware/rag_instruct_test_dataset2_financial_0.1</code>

## Coding Assignment Tasks
**NOTE:** All tasks should resemble those covered in class during the code walkthrough (if you have questions about the use of Google Colab / Jupyter to accomplish these tasks, please refer to the Class Walkthrough and, barring that, ask a GA).  Please refer back to these past materials for guidance on completing the items below.

### Step 1 : Set up your environment.

In [ ]:
# you will be working with some libraries from huggingface: https://huggingface.co/
# these are neither not part of the Python standard library ( https://docs.python.org/3/library/index.html )
# nor bundled with Colab, so you need to install them:
# > datasets (https://pypi.org/project/datasets/)
# > faiss-cpu (https://pypi.org/project/faiss-cpu/)
# > transformers (https://pypi.org/project/transformers/)
# > sentencepiece (https://pypi.org/project/sentencepiece/)

!pip install --quiet datasets faiss-cpu transformers sentencepiece sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# after you have installed these libraries, import them into your notebook
# make sure to import the following: torch, Image (from PIL), faiss, numpy,
# and AutoModel, AutoTokenizer from transformers

# write code below

import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
# instantiate a few variables. first, specify the device as "cpu"
# (you are welcome to set it to "cuda" and work with a GPU but there is no need to do so)

# write code below

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

### Step 2 : Load and inspect your data.



In [ ]:
# import the load_dataset module from the datasets library you installed and
# load the following data from huggingface: https://huggingface.co/datasets/llmware/rag_instruct_test_dataset2_financial_0.1/

# feel free to consult the documentation for guidance if desired: https://huggingface.co/docs/datasets/loading
# NOTE: you may want to load only a portion of the data so that it is easier to work with: https://huggingface.co/docs/datasets/loading#slice-splits

# write code below

from datasets import load_dataset

data = load_dataset("llmware/rag_instruct_test_dataset2_financial_0.1", split="test")

README.md:   0%|          | 0.00/1.94k [00:00<?, ?B/s]

(…)instruct_test_dataset2_financial_0.jsonl:   0%|          | 0.00/131k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# access the features attribute of your dataset object to see its contents

# write code below

data.features

{'query': Value(dtype='string', id=None),
 'answer': Value(dtype='string', id=None),
 'context': Value(dtype='string', id=None),
 'sample_number': Value(dtype='int64', id=None)}

In [ ]:
# access the info attribute of your dataset object to see more detail

# write code below

data.info

DatasetInfo(description='', citation='', homepage='', license='', features={'query': Value(dtype='string', id=None), 'answer': Value(dtype='string', id=None), 'context': Value(dtype='string', id=None), 'sample_number': Value(dtype='int64', id=None)}, post_processed=None, supervised_keys=None, builder_name='json', dataset_name='rag_instruct_test_dataset2_financial_0.1', config_name='default', version=0.0.0, splits={'test': SplitInfo(name='test', num_bytes=124685, num_examples=100, shard_lengths=None, dataset_name='rag_instruct_test_dataset2_financial_0.1')}, download_checksums={'hf://datasets/llmware/rag_instruct_test_dataset2_financial_0.1@8f2e1124e6eae1df7b49f08cde1d9654ca8b72dc/rag_instruct_test_dataset2_financial_0.jsonl': {'num_bytes': 131371, 'checksum': None}}, download_size=131371, post_processing_size=None, dataset_size=124685, size_in_bytes=256056)

In [ ]:
# access the shape attribute of your dataset object to see its shape (as you might expect)

# write code below

data.shape

(100, 4)

### Step 3 : Sample a few rows to familiarize yourself with the data.

In [ ]:
import random

def sample_data(datasetobj):
    i = random.randint(0, datasetobj.shape[0]-1)
    for k in datasetobj.features:
        print('{}: {}\n\n'.format(k, datasetobj[i][k]))

In [ ]:
# sample a few rows to gain some familiarity with the data using the
# provided sample_data method

# write code below

sample_data(data)

query: Which stock has the highest price - (A) River Industries, (B) Smithson Inc., or (C) Johnston Tech?


answer: (A) River Industries


context: The fund currently hold the following positions listed with their current stock prices: Smithson Inc - $19
Johnson Tech - $26
Blitzon Corp - $32
River Industries - $53
Watson Industries - $8
Red Dog LLC - $61


sample_number: 97




In [ ]:
# sample a few rows to gain some familiarity with the data using the
# provided sample_data method

# write code below

sample_data(data)

query: What are operating expenses in first quarter FY24?


answer: $2,508


context: Q2 Fiscal 2024 Summary GAAP ($ in millions, except earnings per share)	Q2 FY24	Q1 FY24	Q2 FY23	Q/Q	Y/Y Revenue	$	13,507	 	$	7,192	 	$	6,704	 	Up 88%	Up 101% Gross margin	 	70.1	%	 	64.6	%	 	43.5	%	Up 5.5 pts	Up 26.6 pts Operating expenses	$	2,662	 	$	2,508	 	$	2,416	 	Up 6%	Up 10% Operating income	$	6,800	 	$	2,140	 	$	499	 	 	Up 218%	Up 1,263% Net income	$	6,188	 	$	2,043	 	$	656	 	Up 203%	Up 843% Diluted earnings per share	$	2.48	 	$	0.82	 	$	0.26	 	Up 202%	Up 854% Non-GAAP ($ in millions, except earnings per share)	Q2 FY24	Q1 FY24	Q2 FY23	Q/Q	Y/Y Revenue	$	13,507	 	$	7,192	 	$	6,704	 	Up 88%	Up 101% Gross margin	 	71.2	%	 	66.8	%	 	45.9	%	Up 4.4 pts	Up 25.3 ptsOperating expenses	$	1,838	 	$	1,750	 	$	1,749	 	Up 5%	Up 5%Operating income	$	7,776	 	$	3,052	 	$	1,325	 	Up 155%	Up 487%Net income	$	6,740	 	$	2,713	 	$	1,292	 	Up 148%	Up 422%Diluted earnings per share	$	2.70	 	$	1.09	 	$	0.51	 	Up 148%	Up

### Step 4 : Create and examine the embeddings for your questions.

In [ ]:
# choose a pretrained model for creating text embeddings
# we will use a variant on modernbert, a model that
# attempts to improve upon the original bert encoder
# you can read about that here: https://huggingface.co/blog/modernbert

# write code below

pretrained_model = "lightonai/modernbert-embed-large"

In [ ]:
# next, instantiate a sentence transformer object using that same pretrained model
# --> what do you think a sentence transformer is doing?

# write code below

embedder = SentenceTransformer(pretrained_model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/227 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/445k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [ ]:
# create and view an embedding for one of the questions
# --> how is this different from the encoding input to an LLM?

# write code below

sample_question = "What are operating expenses in first quarter FY24?"

encoded = embedder.encode(sample_question)

print(encoded)

[0.01445199 0.00557896 0.06528617 ... 0.06925699 0.01034102 0.03620663]


In [ ]:
# create and view another embedding

# write code below

sample_question_2 = "What are the major drivers of revenue growth in the North American region over the last five quarters?"

encoded_2 = embedder.encode(sample_question_2)

print(encoded_2)

[ 0.04076566  0.02011354  0.0349334  ...  0.07660641  0.01300053
 -0.00980639]


In [ ]:
# print the size of your embeddings
# --> why are they the same size?

# write code below

print(encoded.shape)
print(encoded_2.shape)

(1024,)
(1024,)


### Step 5 : Create embeddings for every question in the dataset.

In [ ]:
# you can use the dataset object's map method to apply the embedding operation
# to every question in the data set.

# write code below

data = data.map(lambda cell: {'question_embedding': embedder.encode(cell['query'])})

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
# invoke the add_faiss_index method of the dataset object and specify the
# new column name you created as keyword argument (i.e., column='question_embedding')
# if you are curious about faiss, you can read about it here: https://github.com/facebookresearch/faiss

# write code below

data.add_faiss_index(column='question_embedding')

  0%|          | 0/1 [00:00<?, ?it/s]

Dataset({
    features: ['query', 'answer', 'context', 'sample_number', 'question_embedding'],
    num_rows: 100
})

In [ ]:
# sample another row of data to see your updated dataset

# write code below

sample_data(data)

query: What was gaming revenue in the quarter?


answer: $2.49 billion


context: Gaming Second-quarter revenue was $2.49 billion, up 11% from the previous quarter and up 22% from a year ago. Began shipping the GeForce RTX™ 4060 family of GPUs, bringing to gamers NVIDIA Ada Lovelace architecture and DLSS, starting at $299.Announced NVIDIA Avatar Cloud Engine, or ACE, for Games, a custom AI model foundry service using AI-powered natural language interactions to transform games by bringing intelligence to non-playable characters. Added 35 DLSS games, including Diablo IV, Ratchet & Clank: Rift Apart, Baldur’s Gate 3 and F1 23, as well as Portal: Prelude RTX, a path-traced game made by the community using NVIDIA’s RTX Remix creator tool.Professional Visualization Second-quarter revenue was $379 million, up 28% from the previous quarter and down 24% from a year ago.  Announced three new desktop workstation RTX GPUs based on the Ada Lovelace architecture — NVIDIA RTX 5000, RTX 4500 and RTX 4

In [ ]:
# feel free to repeat the above to get more familiar with the embedding representations
# --> how are these different from the text embeddings you created in week 2?

# write code below

sample_data(data)

query: What was third quarter profit?


answer: 133 million euros.


context: Nokia said it would cut up to 14,000 jobs as part of a cost cutting plan following third quarter earnings that plunged. The Finnish telecommunications giant said that it will reduce its cost base and increase operation efficiency to “address the challenging market environment. The substantial layoffs come after Nokia reported third-quarter net sales declined 20% year-on-year to 4.98 billion euros. Profit over the period plunged by 69% year-on-year to 133 million euros.


sample_number: 79


question_embedding: [0.04776837304234505, 0.012439469806849957, 0.0756845772266388, 0.04722882807254791, 0.00442552613094449, -0.004337124992161989, -0.03420598804950714, 0.030655361711978912, 0.0006144247017800808, -0.04251516982913017, 0.0007648584432899952, -0.005084614735096693, -0.0022046202793717384, 0.0015840984415262938, 0.004058629274368286, 0.014931390061974525, -0.001419226755388081, 0.04235756769776344, -0.0515

### Step 5: Identify nearby, ostensibly similar, questions for a given question.

In [ ]:
def get_nearest_embeddings(dataset_obj, embeddings_col, text, embedding, n=5, printit=True):
    try:
        scores, examples = dataset_obj.get_nearest_examples(embeddings_col, embedding, k=n)
        # similarities = [examples['query'][i] for i in range(0,n)]
        # query_embeddings = embedder.encode(['search_query: {}'.format(dataset_obj['query'][i]) for i in range(0,dataset_obj.shape[0]-1)])
        # doc_embedding = embedder.encode(['search_document: {}'.format(text)])
        # sims = embedder.similarity(query_embeddings, doc_embedding)
        # print(sims)
        if not printit:
            return scores, examples
        print('\033[1mFinding questions similar to:\033[0m "{}"'.format(text))
        print("---------------------------------------------------------------\n")
        for i in range(0,n):
            result_detail = "\033[1mRESULT {}\033[0m ({} distance)".format(i, round(float(scores[i]), ndigits=2))
            print(result_detail)
            print(examples['query'][i])
            print(examples['answer'][i])
            result_footer = "." * (38+len(result_detail))
            print(result_footer)
            print('\n')
    except Exception as e:
        print('Error processing output ({}). Please confirm your inputs are valid and try again.'.format(e))

In [ ]:
# try out a question to see what questions are similar

# write code below

question = "How have revenues changed relative to last year?"

get_nearest_embeddings(data, "question_embedding", question, embedder.encode(question), n=5)

Finding questions similar to: "How have revenues changed relative to last year?"
---------------------------------------------------------------

RESULT 0 (0.24 distance)
What was the increase in revenue from the previous year?
Up 101%
......................................................................


RESULT 1 (0.41 distance)
Did revenue increase more than 50% year-to-year?
Yes, it increased 101% year-to-year.
......................................................................


RESULT 2 (0.44 distance)
What was the increase in professional visualization revenue quarter-to-quarter?
Up 28%
......................................................................


RESULT 3 (0.46 distance)
Did automotive revenue increase year-over-year?
Yes, up 15%.
......................................................................


RESULT 4 (0.47 distance)
What was revenue in second quarter FY23?
$6,704
......................................................................




In [ ]:
question_2 = "What are the major headwinds for growth?"

get_nearest_embeddings(data, "question_embedding", question_2, embedder.encode(question_2), n=5)

Finding questions similar to: "What are the major headwinds for growth?"
---------------------------------------------------------------

RESULT 0 (0.62 distance)
What are the key financial metrics for the outlook for the third quarter?
•Revenue: $16.00 billion, plus or minus 2%;
•Gross Margins: 71.5%-72.%, plus or minor 50 basis points;
•GAAP operating expenses: $2.95 billion;
•Non - GAAP operating expense: $2.00 billion.
......................................................................


RESULT 1 (0.7 distance)
What was the increase in exports?
Up 4.3%
.....................................................................


RESULT 2 (0.75 distance)
What did economists expect for the trade surplus amount?
Deficit of 42.5 billion yen.
......................................................................


RESULT 3 (0.75 distance)
What is a list of the top 3 financial highlights for the quarter?
1. Record revenue of $13.51 billion;
2. Record Data Center revenue of $10.32 billion;
3

### Step 6 : Load LLM to communicate answers

In [ ]:
# import the AutoModelForCausalLM and AutoTokenizer objects from the transformers library
# instantiate a tokenizer and model for the "HuggingFaceTB/SmolLM2-1.7B-Instruct" model you used in week 6

# write code below

from transformers import AutoModelForCausalLM, AutoTokenizer
llm_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(llm_name).to(device)

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/792 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
# create a convenience function to prompt the model

# write code below

def get_llm_output(text):
    messages = [{"role": "user", "content": "{}".format(text)}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
    outputs = llm.generate(inputs, max_new_tokens=150, temperature=0.2, top_p=0.9, do_sample=True)
    return tokenizer.decode(outputs[0])

In [ ]:
# create a function that intakes a question, finds similar questions, and passes
# the answer off to an LLM to communicate with the user

question = "How have revenues changed relative to last year?"

def answer_question(q):
    scores, examples = get_nearest_embeddings(data, 'question_embedding', q, embedder.encode(q), n=5, printit=False)
    top_answer = examples['answer'][0]
    top_answer_score = scores[0]
    p = "The user asked a question '{}' whose answer is '{}'; please restate this answer so that it directly and naturally answers the user's question".format(q, top_answer)
    response = get_llm_output(p).rpartition('<|im_start|>assistant')[2].replace('<|im_end|>','').replace('\n','').strip()
    print('Q>: {}'.format(q))
    print('A>: {}'.format(response))

In [ ]:
# try answering a question

# write code below

answer_question(question)

Q>: How have revenues changed relative to last year?
A>: The revenues have increased by 101% compared to last year.


In [ ]:
# try answering another question

# write code below

question_2 = "What are the worst-performing divisions in the company?"

answer_question(question_2)

Q>: What are the worst-performing divisions in the company?
A>: The worst-performing divisions in the company are Q2 FY24, Q1 FY24, and Q2 FY23, with performance values of $2.70, $1.09, and $0.51, respectively.


### Step 6: Adjust your question answerer to show the ground truth answer and context.

In [ ]:
# update your question answering function to share the ground truth from the data

# write code below

def answer_question(q, full_context=False):
    scores, examples = get_nearest_embeddings(data, 'question_embedding', q, embedder.encode(q), n=5, printit=False)
    top_answer = examples['answer'][0]
    top_answer_score = scores[0]
    p = "The user asked a question '{}' whose answer is '{}'; please restate this answer so that it directly and naturally answers the user's question".format(q, top_answer)
    response = get_llm_output(p).rpartition('<|im_start|>assistant')[2].replace('<|im_end|>','').replace('\n','').strip()
    print('Q>: {}'.format(q))
    print('A>: {}'.format(response))
    if full_context:
      print('\n\n(matched to question-answer pair "{}"/"{}" with context: "{}"'.format(examples['query'][0], top_answer, examples['context'][0]))

In [ ]:
# try a question with this full context provided

# write code below

answer_question(question_2, full_context=True)

Q>: What are the worst-performing divisions in the company?
A>: The worst-performing divisions in the company are Q2 FY24, Q1 FY24, and Q2 FY23, with performance levels of $2.70, $1.09, and $0.51 respectively.
(matched to question-answer pair "What is a list of the diluted earnings per share in Q2 FY24, Q1 FY24, and Q2 FY23?"/"•Q2 FY24 - $2.70;
•Q1 FY24 - $1.09;
•Q2 FY23 - $0.51" with context: "Q2 Fiscal 2024 Summary GAAP ($ in millions, except earnings per share)	Q2 FY24	Q1 FY24	Q2 FY23	Q/Q	Y/Y Revenue	$	13,507	 	$	7,192	 	$	6,704	 	Up 88%	Up 101% Gross margin	 	70.1	%	 	64.6	%	 	43.5	%	Up 5.5 pts	Up 26.6 pts Operating expenses	$	2,662	 	$	2,508	 	$	2,416	 	Up 6%	Up 10% Operating income	$	6,800	 	$	2,140	 	$	499	 	 	Up 218%	Up 1,263% Net income	$	6,188	 	$	2,043	 	$	656	 	Up 203%	Up 843% Diluted earnings per share	$	2.48	 	$	0.82	 	$	0.26	 	Up 202%	Up 854% Non-GAAP ($ in millions, except earnings per share)	Q2 FY24	Q1 FY24	Q2 FY23	Q/Q	Y/Y Revenue	$	13,507	 	$	7,192	 	$	6,704	 	Up 88%	

In [ ]:
# try another question with this full context provided

# write code below

question_3 = "In what direction are overall capital expenditures trending?"

answer_question(question_3, full_context=True)

Q>: In what direction are overall capital expenditures trending?
A>: Overall capital expenditures are trending in the direction of increasing, with revenue growing by 2% and gross margins remaining steady at 71.5%-72.%. GAAP operating expenses have increased by $0.95 billion, while non-GAAP operating expenses have increased by $0.95 billion as well.
(matched to question-answer pair "What are the key financial metrics for the outlook for the third quarter?"/"•Revenue: $16.00 billion, plus or minus 2%;
•Gross Margins: 71.5%-72.%, plus or minor 50 basis points;
•GAAP operating expenses: $2.95 billion;
•Non - GAAP operating expense: $2.00 billion." with context: "Outlook NVIDIA’s outlook for the third quarter of fiscal 2024 is as follows:Revenue is expected to be $16.00 billion, plus or minus 2%. GAAP and non-GAAP gross margins are expected to be 71.5% and 72.5%, respectively, plus or minus 50 basis points.  GAAP and non-GAAP operating expenses are expected to be approximately $2.95 billio

### Step 7 : Adjust your question answerer to decline to answer questions when unable to find a strong match.

In [ ]:
# update your question answering function to avoid uncertainty

# write code below

def answer_question(q, full_context=False):
    scores, examples = get_nearest_embeddings(data, 'question_embedding', q, embedder.encode(q), n=5, printit=False)
    top_answer = examples['answer'][0]
    top_answer_score = scores[0]
    if top_answer_score > 0.5:
        print("I'm sorry, but there may not be data to support answering that question.  Perhaps you would be interested in these similar questions:\n")
        for i in range(0,5):
            print('> {}'.format(examples['query'][i]))
        return None
    p = "The user asked a question '{}' whose answer is '{}'; please restate this answer so that it directly and naturally answers the user's question".format(q, top_answer)
    response = get_llm_output(p).rpartition('<|im_start|>assistant')[2].replace('<|im_end|>','').replace('\n','').strip()
    print('Q>: {}'.format(q))
    print('A>: {}'.format(response))
    if full_context:
      print('\n\n(matched to question-answer pair "{}"/"{}" with context: "{}"'.format(examples['query'][0], top_answer, examples['context'][0]))

In [ ]:
# try another question with this new feature included

# write code below

question_3 = "In what direction are overall capital expenditures trending?"

answer_question(question_3, full_context=True)

I'm sorry, but there may not be data to support answering that question.  Perhaps you would be interested in these similar questions:

> What are the key financial metrics for the outlook for the third quarter?
> Did automotive revenue increase year-over-year?
> What was the increase in revenue from the previous year?
> What was the amount of free cash flow in the quarter?
> Did revenue increase more than 50% year-to-year?


In [ ]:
# try another question with this new feature included

# write code below

question_4 = "How much have profit margins increased?"

answer_question(question_4, full_context=True)

Q>: How much have profit margins increased?
A>: The profit margins have increased by 101%.


(matched to question-answer pair "What was the increase in revenue from the previous year?"/"Up 101%" with context: "NVIDIA Announces Financial Results for Second Quarter Fiscal 2024 August 23, 20231	Record revenue of $13.51 billion, up 88% from Q1, up 101% from year ago2	Record Data Center revenue of $10.32 billion, up 141% from Q1, up 171% from year agoNVIDIA (NASDAQ: NVDA) today reported revenue for the second quarter ended July 30, 2023, of $13.51 billion, up 101% from a year ago and up 88% from the previous quarter.GAAP earnings per diluted share for the quarter were $2.48, up 854% from a year ago and up 202% from the previous quarter. Non-GAAP earnings per diluted share were $2.70, up 429% from a year ago and up 148% from the previous quarter. A new computing era has begun. Companies worldwide are transitioning from general-purpose to accelerated computing and generative AI,” said Jensen 

### Step 8 : Compare these outputs to those directly from the language model.

In [ ]:
def direct_to_llm(q):
    p = 'Please provide a direct and natural answer to the question: "{}"'.format(q)
    response = get_llm_output(p).rpartition('<|im_start|>assistant')[2].replace('<|im_end|>','').replace('\n','').strip()
    print(response)

In [ ]:
direct_to_llm(question_4)

As an AI, I don't have real-time data or the ability to access current financial data. I recommend checking the latest financial reports or news from reliable sources to get the most accurate information about profit margins.


### Step 9 : Compare these outputs to those directly from the language model (with context).

In [ ]:
def direct_to_llm(q, context):
    p = 'Please provide a direct and natural answer to the question: "{}" given the following context: "{}"'.format(q, context)
    response = get_llm_output(p).rpartition('<|im_start|>assistant')[2].replace('<|im_end|>','').replace('\n','').strip()
    print(response)

In [ ]:
c = "NVIDIA Announces Financial Results for Second Quarter Fiscal 2024 August 23, 20231	Record revenue of $13.51 billion, up 88% from Q1, up 101% from year ago2	Record Data Center revenue of $10.32 billion, up 141% from Q1, up 171% from year agoNVIDIA (NASDAQ: NVDA) today reported revenue for the second quarter ended July 30, 2023, of $13.51 billion, up 101% from a year ago and up 88% from the previous quarter.GAAP earnings per diluted share for the quarter were $2.48, up 854% from a year ago and up 202% from the previous quarter. Non-GAAP earnings per diluted share were $2.70, up 429% from a year ago and up 148% from the previous quarter. A new computing era has begun. Companies worldwide are transitioning from general-purpose to accelerated computing and generative AI,” said Jensen Huang, founder and CEO of NVIDIA.  NVIDIA GPUs connected by our Mellanox networking and switch technologies and running our CUDA AI software stack make up the computing infrastructure of generative AI.  During the quarter, major cloud service providers announced massive NVIDIA H100 AI infrastructures. Leading enterprise IT system and software providers announced partnerships to bring NVIDIA AI to every industry. The race is on to adopt generative AI,” he said. During the second quarter of fiscal 2024, NVIDIA returned $3.38 billion to shareholders in the form of 7.5 million shares repurchased for $3.28 billion, and cash dividends. As of the end of the second quarter, the company had $3.95 billion remaining under its share repurchase authorization. On August 21, 2023, the Board of Directors approved an additional $25.00 billion in share repurchases, without expiration. NVIDIA plans to continue share repurchases this fiscal year.NVIDIA will pay its next quarterly cash dividend of $0.04 per share on September 28, 2023, to all shareholders of record on September 7, 2023."

In [ ]:
direct_to_llm(question_4, c)

Profit margins have increased significantly for NVIDIA. The company reported a record revenue of $13.51 billion, up 88% from the first quarter and 101% from the previous year. Additionally, the company's Data Center revenue reached a record $10.32 billion, up 141% from the first quarter and 171% from the previous year. These figures demonstrate a substantial growth in both revenue and Data Center revenue, indicating a strong performance by NVIDIA in the second quarter of fiscal 2024.
